# S08 — Exercises: Testing, Type Hints, Exceptions & Defensive Programming

**Python for Applied Physics** | Master in Applied Physics

## Workflow for this session

Each exercise has two deliverables:

1. **Notebook cell** — write and validate the code here first
2. **Repo file** — copy the final version to `tests/test_*.py` and run `pytest` from the terminal

At the end of the session your repo should have:
```
tests/
├── conftest.py
├── test_optics.py
├── test_pulses.py
├── test_pulse_classes.py
├── test_beam_classes.py
└── test_io.py
```
and `pytest tests/ -v` should pass all tests.

| Symbol | Difficulty |
|--------|------------|
| 🟢 | Accessible |
| 🟡 | Intermediate |
| 🔴 | Challenging |

In [ ]:
from __future__ import annotations

import numpy as np
import pytest
import warnings
import logging
from typing import Optional, Union

print("Setup complete")

---
## Exercise 1 🟢 — Type-annotate `shared/optics.py` functions

The functions below are the current state of `shared/optics.py` (simplified). Add complete type annotations to all function signatures and return types.

**Tasks:**
1. Add type hints to all parameters and return values.
2. Use `Optional` where a parameter can be `None`.
3. Run `mypy` on the annotated file from the terminal and fix any errors.
4. Verify that calling a function with a wrong type does **not** raise a runtime error (confirm that hints are documentation only).

**Repo deliverable:** update `shared/optics.py` with the annotations and commit.

In [ ]:
# Current shared/optics.py functions — add type hints

def gaussian_intensity(r, w, P):   # ← add hints
    """Radial intensity profile of a Gaussian beam (W/m²)."""
    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)


def gaussian_field_2d(x, y, w, E0=1.0):   # ← add hints
    """2D Gaussian field via broadcasting."""
    X = x[np.newaxis, :]
    Y = y[:, np.newaxis]
    return E0 * np.exp(-(X**2 + Y**2) / w**2)


def free_space(d):   # ← add hints
    """ABCD matrix for free-space propagation."""
    return np.array([[1.0, d], [0.0, 1.0]])


def thin_lens(f):   # ← add hints
    """ABCD matrix for a thin lens."""
    return np.array([[1.0, 0.0], [-1.0/f, 1.0]])


def fresnel_rs(n1, n2, theta_i):   # ← add hints
    """s-polarisation Fresnel reflection coefficient."""
    sin_t = n1 / n2 * np.sin(theta_i)
    cos_t = np.sqrt(np.clip(1 - sin_t**2, 0, None))
    cos_i = np.cos(theta_i)
    return (n1*cos_i - n2*cos_t) / (n1*cos_i + n2*cos_t)


# --- Assertions ---
# 1. Verify hints don't break execution
r = np.linspace(-1e-3, 1e-3, 64)
I = gaussian_intensity(r, 300e-6, 1.0)
assert I.shape == r.shape

# 4. Wrong type does NOT raise at runtime
result = gaussian_intensity(r, '300e-6', 1.0)   # type: ignore
print("Type hints don't enforce at runtime — no error raised with wrong type")

print("\nAnnotated functions (add hints to the signatures above):")
print("  gaussian_intensity(r: np.ndarray, w: float, P: float) -> np.ndarray")
print("  gaussian_field_2d(x: np.ndarray, y: np.ndarray, w: float, E0: float = 1.0) -> np.ndarray")
print("  free_space(d: float) -> np.ndarray")
print("  thin_lens(f: float) -> np.ndarray")
print("  fresnel_rs(n1: float, n2: float, theta_i: float | np.ndarray) -> float | np.ndarray")

**Terminal commands:**
```bash
pip install mypy
mypy shared/optics.py --ignore-missing-imports
```

---
## Exercise 2 🟢 — Unit tests for `gaussian_intensity`

Write a complete test suite for `gaussian_intensity` covering all physically meaningful cases.

**Tasks:**
1. Test peak value at `r=0`.
2. Test value at `r=w` (should be `I₀·e⁻²`).
3. Test radial symmetry: `I(r) == I(-r)`.
4. Test integrated power equals `P` (within 0.1% for a fine grid).
5. Test that doubling `P` doubles the peak.
6. Test output shape matches input shape.

**Repo deliverable:** save as `tests/test_optics.py`.

In [ ]:
# Write your tests here, then copy to tests/test_optics.py

def gaussian_intensity(r: np.ndarray, w: float, P: float) -> np.ndarray:
    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)


# 1. Peak at r=0
def test_gaussian_intensity_peak():
    # YOUR CODE HERE
    pass

# 2. Value at r=w
def test_gaussian_intensity_at_w():
    # YOUR CODE HERE
    pass

# 3. Radial symmetry
def test_gaussian_intensity_symmetry():
    # YOUR CODE HERE
    pass

# 4. Integrated power
def test_gaussian_intensity_integrated_power():
    # YOUR CODE HERE
    pass

# 5. Linearity in P
def test_gaussian_intensity_linear_in_power():
    # YOUR CODE HERE
    pass

# 6. Output shape
def test_gaussian_intensity_shape():
    # YOUR CODE HERE
    pass


# Run all tests
for fn in [test_gaussian_intensity_peak, test_gaussian_intensity_at_w,
           test_gaussian_intensity_symmetry, test_gaussian_intensity_integrated_power,
           test_gaussian_intensity_linear_in_power, test_gaussian_intensity_shape]:
    fn()
    print(f"  {fn.__name__} ✓")

print("\nAll tests passed ✓")
print("→ Copy these functions to tests/test_optics.py and run: pytest tests/test_optics.py -v")

---
## Exercise 3 🟡 — Parametrized TBP test

The time-bandwidth product of a transform-limited Gaussian pulse is always $\approx 0.4413$, independent of duration.

**Tasks:**
1. Write `compute_tbp(tau_fs: float) -> float` that computes the numerical TBP.
2. Write a single parametrized test covering `tau_fs` in `[30, 50, 80, 100, 150, 200, 500]`.
3. Use `pytest.mark.parametrize` with an `id` for each case.
4. Also parametrize over two wavelengths (800 nm and 1030 nm) and verify TBP is wavelength-independent.

**Repo deliverable:** save as `tests/test_pulses.py`.

In [ ]:
TBP_LIMIT = 2 * np.log(2) / np.pi


def compute_tbp(tau_fs: float, wavelength_nm: float = 800.0) -> float:
    """Numerical TBP for a transform-limited Gaussian pulse."""
    # YOUR CODE HERE
    pass


@pytest.mark.parametrize('tau_fs', [30, 50, 80, 100, 150, 200, 500])
def test_tbp_vs_duration(tau_fs: float):
    """TBP is invariant to pulse duration."""
    # YOUR CODE HERE
    pass


@pytest.mark.parametrize('wavelength_nm', [800.0, 1030.0])
@pytest.mark.parametrize('tau_fs', [100, 200])
def test_tbp_vs_wavelength(tau_fs: float, wavelength_nm: float):
    """TBP is wavelength-independent."""
    # YOUR CODE HERE
    pass


# Run manually in notebook
assert compute_tbp is not None
for tau_fs in [30, 50, 80, 100, 150, 200, 500]:
    tbp = compute_tbp(float(tau_fs))
    ok  = abs(tbp - TBP_LIMIT) / TBP_LIMIT < 0.01
    print(f"  tau={tau_fs:4d} fs: TBP={tbp:.4f} {'✓' if ok else '✗'}")

print("\n→ Copy to tests/test_pulses.py and run: pytest tests/test_pulses.py -v")

---
## Exercise 4 🟡 — Test suite for `GaussianPulse`

Write a complete test suite for `GaussianPulse` from `shared/pulse_classes.py`.

**Tasks:**
1. Write `conftest.py` with a `standard_pulse` fixture and a `time_axis` fixture.
2. Test `__repr__` contains key information.
3. Test `intensity` peak and 1/e decay.
4. Test that `PulseError` is raised for `tau ≤ 0`, `wavelength ≤ 0`, `energy < 0`.
5. Test `fwhm` and `bandwidth` give the correct TBP.
6. Test `spectrum()` returns arrays of the correct shape and non-negative values.

**Repo deliverable:** save as `tests/test_pulse_classes.py` and `tests/conftest.py`.

In [ ]:
# Minimal GaussianPulse for this exercise
class PulseError(ValueError):
    def __init__(self, parameter, value, reason):
        self.parameter = parameter; self.value = value
        super().__init__(f"{parameter}={value}: {reason}")

class GaussianPulse:
    c = 3e8
    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        if tau <= 0:        raise PulseError('tau', tau, 'must be positive')
        if wavelength <= 0: raise PulseError('wavelength', wavelength, 'must be positive')
        if energy < 0:      raise PulseError('energy', energy, 'must be non-negative')
        self._tau = tau; self._wavelength = wavelength; self._energy = energy

    @property
    def tau(self) -> float: return self._tau
    @property
    def wavelength(self) -> float: return self._wavelength
    @property
    def energy(self) -> float: return self._energy
    @property
    def fwhm(self) -> float: return 2 * np.sqrt(np.log(2)) * self._tau
    @property
    def bandwidth(self) -> float: return np.sqrt(np.log(2)) / (np.pi * self._tau)

    def intensity(self, t: np.ndarray) -> np.ndarray:
        return np.exp(-t**2 / self._tau**2)

    def spectrum(self, t: np.ndarray):
        N = len(t); dt = t[1] - t[0]
        E_f = np.fft.fftshift(np.fft.fft(np.sqrt(self.intensity(t))))
        freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
        return freq, np.abs(E_f)**2

    def __repr__(self) -> str:
        return (f"GaussianPulse(tau={self._tau*1e15:.1f} fs, "
                f"wavelength={self._wavelength*1e9:.0f} nm, "
                f"energy={self._energy*1e6:.1f} µJ)")


# conftest.py content
@pytest.fixture
def standard_pulse() -> GaussianPulse:
    # YOUR CODE HERE
    pass

@pytest.fixture
def time_axis() -> np.ndarray:
    # YOUR CODE HERE
    pass


# test_pulse_classes.py content
def test_repr_contains_tau(standard_pulse):
    # YOUR CODE HERE
    pass

def test_intensity_peak(standard_pulse):
    # YOUR CODE HERE
    pass

def test_intensity_at_tau(standard_pulse):
    """I(tau) = exp(-1)."""
    # YOUR CODE HERE
    pass

def test_raises_negative_tau():
    # YOUR CODE HERE
    pass

def test_raises_zero_wavelength():
    # YOUR CODE HERE
    pass

def test_raises_negative_energy():
    # YOUR CODE HERE
    pass

def test_zero_energy_valid():
    # YOUR CODE HERE
    pass

def test_tbp(standard_pulse, time_axis):
    """fwhm * bandwidth ≈ TBP_LIMIT."""
    # YOUR CODE HERE
    pass

def test_spectrum_shape(standard_pulse, time_axis):
    # YOUR CODE HERE
    pass

def test_spectrum_non_negative(standard_pulse, time_axis):
    # YOUR CODE HERE
    pass


# Run with simulated fixtures
sp = standard_pulse() if callable(standard_pulse) else GaussianPulse(100e-15, 800e-9, 50e-6)
ta = time_axis() if callable(time_axis) else (np.arange(4096)-2048)*5e-15

# If fixtures return None (not implemented yet), use defaults
sp = sp or GaussianPulse(100e-15, 800e-9, 50e-6)
ta = ta if ta is not None else (np.arange(4096)-2048)*5e-15

tests_to_run = [
    (test_repr_contains_tau,    (sp,)),
    (test_intensity_peak,       (sp,)),
    (test_intensity_at_tau,     (sp,)),
    (test_raises_negative_tau,  ()),
    (test_raises_zero_wavelength, ()),
    (test_raises_negative_energy, ()),
    (test_zero_energy_valid,    ()),
    (test_tbp,                  (sp, ta)),
    (test_spectrum_shape,       (sp, ta)),
    (test_spectrum_non_negative,(sp, ta)),
]

for fn, args in tests_to_run:
    try:
        fn(*args)
        print(f"  {fn.__name__} ✓")
    except (AssertionError, TypeError, NotImplementedError) as e:
        print(f"  {fn.__name__} — not yet implemented")

print("\n→ Copy to tests/test_pulse_classes.py and tests/conftest.py")

---
## Exercise 5 🟡 — Test the `OpticalElement` hierarchy

Write integration tests for the `OpticalElement` hierarchy from `shared/beam_classes.py`.

**Tasks:**
1. Test `FreeSpace(0)` gives the identity matrix.
2. Test `ThinLens(f) @ FreeSpace(f) @ ThinLens(f)` collimates a point source.
3. Test that the determinant of any single element matrix equals 1 (det = 1 for lossless systems).
4. Test that `@` composition is associative.
5. Test `FreeSpace(d)` raises `ValueError` for `d < 0`.
6. Test `ThinLens(f)` raises `ValueError` for `f == 0`.

**Repo deliverable:** save as `tests/test_beam_classes.py`.

In [ ]:
# Minimal element classes for this exercise
class OpticalElement:
    def matrix(self) -> np.ndarray: raise NotImplementedError
    def propagate(self, ray: np.ndarray) -> np.ndarray: return self.matrix() @ ray
    def __matmul__(self, other):
        if isinstance(other, OpticalElement):
            return _Composed(self.matrix() @ other.matrix())
        return NotImplemented

class _Composed(OpticalElement):
    def __init__(self, M): self._M = M
    def matrix(self): return self._M

class FreeSpace(OpticalElement):
    def __init__(self, d: float) -> None:
        if d < 0: raise ValueError(f"d must be non-negative, got {d}")
        self.d = d
    def matrix(self) -> np.ndarray: return np.array([[1.0, self.d],[0.0, 1.0]])

class ThinLens(OpticalElement):
    def __init__(self, f: float) -> None:
        if f == 0: raise ValueError("f cannot be zero")
        self.f = f
    def matrix(self) -> np.ndarray: return np.array([[1.0, 0.0],[-1/self.f, 1.0]])

class Mirror(OpticalElement):
    def __init__(self, R: float) -> None: self.R = R
    def matrix(self) -> np.ndarray: return np.array([[1.0,0.0],[-2/self.R, 1.0]])


# Write tests here, then copy to tests/test_beam_classes.py

def test_free_space_identity():
    # YOUR CODE HERE
    pass

def test_collimation():
    """f-f system collimates a point source: output angle = 0."""
    # YOUR CODE HERE
    pass

@pytest.mark.parametrize('element', [
    FreeSpace(0.1), ThinLens(0.2), Mirror(0.5)
])
def test_det_equals_one(element):
    """Lossless elements have det(M) = 1."""
    # YOUR CODE HERE
    pass

def test_composition_associativity():
    # YOUR CODE HERE
    pass

def test_free_space_negative_raises():
    # YOUR CODE HERE
    pass

def test_thin_lens_zero_raises():
    # YOUR CODE HERE
    pass


# Run
for fn, args in [
    (test_free_space_identity,     ()),
    (test_collimation,             ()),
    (test_composition_associativity, ()),
    (test_free_space_negative_raises, ()),
    (test_thin_lens_zero_raises,   ()),
]:
    try:
        fn(*args); print(f"  {fn.__name__} ✓")
    except (AssertionError, TypeError, NotImplementedError):
        print(f"  {fn.__name__} — not yet implemented")

for el in [FreeSpace(0.1), ThinLens(0.2), Mirror(0.5)]:
    try:
        test_det_equals_one(el); print(f"  test_det_equals_one({el.__class__.__name__}) ✓")
    except (AssertionError, TypeError, NotImplementedError):
        print(f"  test_det_equals_one — not yet implemented")

print("\n→ Copy to tests/test_beam_classes.py")

---
## Exercise 6 🟡 — Custom exceptions + defensive guards in `shared/io.py`

**Tasks:**
1. Define `IOError_` (avoid shadowing builtin) or use a custom `PulseIOError(PhysicsError, IOError)` exception.
2. Add guard clauses to `save_pulse_hdf5`: raise `ValueError` if `t` and `E_t` have different lengths; raise `ValueError` if `path` doesn't end in `.h5`.
3. Add a `DeprecationWarning` if a caller passes `compress=True` (old parameter name) instead of `compression='gzip'`.
4. Add `logging` calls: `DEBUG` before writing, `INFO` after success, `WARNING` if the file already exists.
5. Write tests for all guards and warnings.

**Repo deliverable:** update `shared/io.py` and save tests as `tests/test_io.py`.

In [ ]:
import h5py, tempfile, os

logger_io = logging.getLogger('shared.io')


class PhysicsError(Exception): pass

class PulseIOError(PhysicsError, OSError):
    """Raised for I/O errors specific to pulse data."""


def save_pulse_hdf5(
    path: str,
    t: np.ndarray,
    E_t: np.ndarray,
    compression: str = 'gzip',
    compress: Optional[bool] = None,   # deprecated
    **metadata,
) -> None:
    """
    Save a pulse field and metadata to HDF5.

    Parameters
    ----------
    path        : str         — output file path (must end in .h5)
    t           : np.ndarray  — time array (s)
    E_t         : np.ndarray  — complex field envelope
    compression : str         — HDF5 compression filter
    **metadata  — stored as HDF5 group attributes
    """
    # Guard clauses
    # YOUR CODE HERE — validate path extension, array lengths

    # Deprecation warning for old 'compress' parameter
    # YOUR CODE HERE

    # Logging
    # YOUR CODE HERE

    with h5py.File(path, 'w') as f:
        grp = f.create_group('pulse')
        grp.create_dataset('t',   data=t,   compression=compression)
        grp.create_dataset('E_t', data=E_t, compression=compression)
        grp.create_dataset('I_t', data=np.abs(E_t)**2)
        for key, value in metadata.items():
            grp.attrs[key] = value

    # YOUR CODE HERE — INFO log


# Tests
N = 512; dt = 5e-15
t_test  = (np.arange(N) - N//2) * dt
E_test  = np.exp(-t_test**2 / (2*(100e-15)**2))

def test_save_wrong_extension():
    with pytest.raises(ValueError, match=r'\.h5'):
        save_pulse_hdf5('output.txt', t_test, E_test)

def test_save_mismatched_lengths():
    with pytest.raises(ValueError, match='length'):
        save_pulse_hdf5('/tmp/test.h5', t_test, E_test[:10])

def test_save_deprecation_warning():
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, 'pulse.h5')
        with pytest.warns(DeprecationWarning):
            save_pulse_hdf5(path, t_test, E_test, compress=True)

def test_save_roundtrip():
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, 'pulse.h5')
        save_pulse_hdf5(path, t_test, E_test, tau_fs=100.0)
        with h5py.File(path, 'r') as f:
            t_back  = f['pulse']['t'][:]
            E_back  = f['pulse']['E_t'][:]
            tau_attr = f['pulse'].attrs['tau_fs']
        assert np.allclose(t_back, t_test)
        assert np.allclose(E_back, E_test)
        assert tau_attr == pytest.approx(100.0)


for fn in [test_save_wrong_extension, test_save_mismatched_lengths,
           test_save_deprecation_warning, test_save_roundtrip]:
    try:
        fn(); print(f"  {fn.__name__} ✓")
    except Exception as e:
        print(f"  {fn.__name__} — {type(e).__name__}: {e}")

print("\n→ Copy tests to tests/test_io.py, update shared/io.py")

---
## Exercise 7 🔴 — Achieve 80% coverage on `shared/`

**Tasks:**
1. Run `pytest tests/ --cov=shared --cov-report=term-missing` and note the current coverage.
2. Identify the uncovered lines (the `Missing` column).
3. Write additional tests targeting the uncovered branches — focus on: edge cases (N=1 array, zero power, very large values), error paths not yet tested, code branches in `if/else` blocks.
4. Re-run coverage until `shared/` reaches ≥ 80%.
5. Add `# pragma: no cover` to any code that is intentionally untestable (e.g. debug-only branches).

**Repo deliverable:** add gap-filling tests to the appropriate `test_*.py` files.

In [ ]:
# Terminal workflow — run these commands from your repo root

COVERAGE_COMMANDS = """
# 1. Run with coverage
pytest tests/ --cov=shared --cov-report=term-missing -v

# 2. Generate HTML report
pytest tests/ --cov=shared --cov-report=html
open htmlcov/index.html   # macOS
# xdg-open htmlcov/index.html   # Linux

# 3. Fail if coverage drops below threshold
pytest tests/ --cov=shared --cov-fail-under=80
"""

print(COVERAGE_COMMANDS)

# Gap-filling test examples — add more based on your coverage report

def test_gaussian_intensity_single_point():
    """Works on a scalar wrapped in array."""
    # YOUR CODE HERE
    pass

def test_gaussian_intensity_zero_power():
    """Zero power gives zero intensity everywhere."""
    # YOUR CODE HERE
    pass

def test_free_space_zero_distance():
    """d=0 gives identity — edge case."""
    # YOUR CODE HERE
    pass

def test_pulse_very_short():
    """tau=1e-16 (100 as) is valid."""
    # YOUR CODE HERE
    pass


for fn in [test_gaussian_intensity_single_point,
           test_gaussian_intensity_zero_power,
           test_free_space_zero_distance,
           test_pulse_very_short]:
    try:
        fn(); print(f"  {fn.__name__} ✓")
    except (AssertionError, TypeError, NotImplementedError):
        print(f"  {fn.__name__} — not yet implemented")

print("\n→ Add these and additional gap-filling tests to the appropriate test_*.py files")
print("→ Target: pytest tests/ --cov=shared --cov-fail-under=80")

---
## Wrap-up checklist

### Notebook
- [ ] All 7 exercises: assertions pass

### Repository
- [ ] `tests/conftest.py` — `standard_pulse` and `time_axis` fixtures
- [ ] `tests/test_optics.py` — 6 tests for `gaussian_intensity`
- [ ] `tests/test_pulses.py` — parametrized TBP tests
- [ ] `tests/test_pulse_classes.py` — 10 tests for `GaussianPulse`
- [ ] `tests/test_beam_classes.py` — 6 tests for `OpticalElement` hierarchy
- [ ] `tests/test_io.py` — 4 tests for `save_pulse_hdf5`
- [ ] `pytest tests/ -v` — all tests green
- [ ] `pytest tests/ --cov=shared --cov-fail-under=80` — coverage ≥ 80%
- [ ] `shared/optics.py` — fully type-annotated
- [ ] `shared/io.py` — defensive guards and logging added
- [ ] `git add tests/ shared/ && git commit -m "S08: add test suite and type annotations"`

**Next session:** S09 — Performance & FFI